# Residential vs non-residential buildings classification

OSM data is incomplete when comes to residential / non-residential classification. Large share of footprints has just a non-specific tag "building".

Let's train a model to 

In [2]:
import geopandas as gpd
import osmnx as ox
import pandas as pd

data_folder = 'source_data/'

In [3]:
buildings = gpd.read_file(f'{data_folder}/building_heights.geojson')
districts = gpd.read_file(f'{data_folder}/districts_with_stats.geojson')

In [4]:
buildings.head(1)

,height,building_type,name,function,lat,lon,landuse,geometry
0,5.671275,yes,None,None,20.485226,44.937775,farmyard,"POLYGON ((20.48522 44.93792, 20.48541 44.9377,..."


In [5]:
districts_proj = ox.projection.project_gdf(districts)
buildings_proj = ox.projection.project_gdf(buildings)
districts_proj['population_density'] = districts_proj['population_2024'] / districts_proj.geometry.area
districts_proj['income_density'] = districts_proj['monthly_income_2025'] * districts_proj['population_density']  # TODO: think it through? what is this param?
districts_proj.head(2)

,name,name_en,name_norm,population_2024,monthly_income_2025,geometry,population_density,income_density
0,Gradska opština Stari grad,Stari Grad Urban Municipality,stari grad,44098,187785,"POLYGON ((456012.577 4963726.03, 456086.369 49...",0.008204,1540.622500
1,Gradska opština Grocka,Grocka Urban Municipality,grocka,83012,97460,"POLYGON ((462987.133 4947598.549, 462988.916 4...",0.000277,27.016093


In [6]:
buildings_proj = buildings_proj.sjoin(districts_proj[['geometry', 'population_density', 'income_density']], predicate='within')

## Some feature engineering


In [7]:
import numpy as np

In [8]:
buildings_proj['area'] = buildings_proj.geometry.area
buildings_proj['perimeter'] = buildings_proj.geometry.length
buildings_proj['volume'] = buildings_proj.area * buildings_proj.perimeter
buildings_proj['slenderness'] = buildings_proj.height / np.sqrt(buildings_proj.area)
buildings_proj[buildings_proj['slenderness'] > 10] = np.nan
buildings_proj['compactness'] = 4 * np.pi * buildings_proj.area / (buildings_proj.perimeter ** 2)
buildings_proj.head(3)

,height,building_type,name,function,lat,lon,landuse,geometry,index_right,population_density,income_density,area,perimeter,volume,slenderness,compactness
0,5.671275,yes,None,None,20.485226,44.937775,farmyard,"POLYGON ((459383.657 4976183.351, 459398.318 4...",11.0,0.000409,50.666154,466.166466,89.959844,41936.262676,0.262670,0.723858
1,8.052044,yes,None,None,20.485714,44.937992,farmyard,"POLYGON ((459421.482 4976207.996, 459437.618 4...",11.0,0.000409,50.666154,491.851058,93.379530,45928.820735,0.363069,0.708827
2,5.712940,yes,None,None,20.486197,44.938209,farmyard,"POLYGON ((459445.709 4976223.45, 459460.677 49...",11.0,0.000409,50.666154,532.033014,96.000945,51075.672210,0.247680,0.725433


In [9]:
numeric_features = ['height', 'area', 'perimeter', 'volume', 'slenderness', 'compactness', 'population_density', 'income_density']
categorical_features = ['function', 'landuse']

## Construct the target 


In [10]:
is_residential = buildings_proj.building_type.isin(['house', 'apartments', 'dormitory', 'residential'])
is_non_residential = buildings_proj.building_type.isin(['office', 'commercial', 'hotel', 'retail', 'industrial'])

unknown = (~is_residential) & (~is_non_residential)

In [11]:
buildings_proj.loc[:, 'target'] = is_residential.astype(int)
buildings_proj.loc[unknown, 'target'] = np.nan

In [12]:
dataset = buildings_proj[~unknown]
to_predict = buildings_proj[unknown]

In [18]:
print(dataset.shape, to_predict.shape)

(171374, 17) (199663, 17)


## Dataset Preprocessing and Normalization


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier


In [20]:
numeric_features = [
    # 'height',
    'area',
    # 'perimeter',
    'volume',
    # 'slenderness',
    # 'compactness',
    # 'income_density',
    'population_density'
]
categorical_features = [
    'function',
    'landuse'
]

In [21]:
print(f'Categorical features: {categorical_features}')
print(f'Numerical features: {numeric_features}')
all_features = categorical_features + numeric_features

X = dataset[all_features]
y = dataset.target

for feature in categorical_features:
    X.loc[:,feature] = X[feature].fillna('')


Categorical features: ['function', 'landuse']
Numerical features: ['area', 'volume', 'population_density']


In [22]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler()),
        ('scaler_norm', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='')),
        # ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)
transformers = [
    ('cat', categorical_transformer, categorical_features),
    ('num', numeric_transformer, numeric_features),
]

In [23]:

X.head(3)

,function,landuse,area,volume,population_density
5,,residential,202.979857,11767.672242,0.000409
8,,residential,129.782262,5967.785112,0.000409
9,,residential,193.412989,10768.474177,0.000409


In [24]:
preprocess = ColumnTransformer(
    transformers=transformers,
    remainder='passthrough'
)
X_norm = pd.DataFrame(preprocess.fit_transform(X), columns=X.columns)
X_norm

,function,landuse,area,volume,population_density
0,,residential,0.03988,-0.019755,-0.607559
1,,residential,-0.079062,-0.028975,-0.607559
2,,residential,0.024334,-0.021344,-0.607559
3,,residential,0.106842,-0.014145,-0.607559
4,,residential,-0.075733,-0.028719,-0.607559
...,...,...,...,...,...
171369,,residential,-0.169911,-0.03432,-0.607559
171370,,,0.271735,0.004633,1.030372
171371,,,-0.166738,-0.03425,1.030372
171372,,,-0.206572,-0.036117,0.349371


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.15, random_state=42)
X_test, X_eval, y_test, y_eval = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

## Train the model

In [26]:
model = CatBoostClassifier(
    n_estimators=1400,
    l2_leaf_reg=8,
    learning_rate=0.02,
    random_strength=1,
    max_depth=12,
    random_state=42,
    cat_features=categorical_features,
    eval_metric='AUC',
    custom_metric=['AUC'],
    use_best_model=True,
    class_weights=[1.0, 2.6]
)

model.fit(X_train, y_train, eval_set=(X_eval, y_eval), verbose=200)

0:	test: 0.8644761	best: 0.8644761 (0)	total: 136ms	remaining: 3m 10s
200:	test: 0.9102863	best: 0.9102863 (200)	total: 12s	remaining: 1m 11s
400:	test: 0.9122995	best: 0.9123002 (398)	total: 22.9s	remaining: 57.1s
600:	test: 0.9128782	best: 0.9128786 (596)	total: 33s	remaining: 43.9s
800:	test: 0.9132181	best: 0.9132181 (800)	total: 43.3s	remaining: 32.4s
1000:	test: 0.9132652	best: 0.9132706 (959)	total: 51.8s	remaining: 20.6s
1200:	test: 0.9133051	best: 0.9133137 (1177)	total: 59.8s	remaining: 9.92s
1399:	test: 0.9132969	best: 0.9133137 (1177)	total: 1m 7s	remaining: 0us

bestTest = 0.9133137314
bestIteration = 1177

Shrink model to first 1178 iterations.


CatBoostClassifier(cat_features=['function', 'landuse'], class_weights=[1.0, 2.6], custom_metric=['AUC'], eval_metric='AUC', l2_leaf_reg=8, learning_rate=0.02, max_depth=12, n_estimators=1400, random_state=42, random_strength=1, use_best_model=True)

In [27]:
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score

probs = model.predict_proba(X_eval)[:, 1]

thresholds = np.linspace(0.1, 0.95, 100)
best_t, best_f1 = 0, 0

for t in thresholds:
    preds = (probs > t).astype(int)
    f1 = f1_score(y_eval.astype(int), preds, pos_label=0)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(best_t, best_f1)

0.8984848484848484 0.5652173913043478


In [28]:
from sklearn.metrics import precision_recall_curve

In [29]:
y_test

109723    1.0
277898    1.0
12375     1.0
24024     1.0
165747    1.0
         ... 
129334    1.0
133245    1.0
117815    1.0
258531    1.0
137994    1.0
Name: target, Length: 12853, dtype: float64

In [30]:
confidence = best_t

y_pred_proba = model.predict_proba(X_test)
y_pred = (y_pred_proba[:, 1] > confidence).astype(int)
precision, recall, f1, support = precision_recall_fscore_support(y_test.astype(int), y_pred, labels=[0, 1])

print(f"precision (neg, pos): {precision}")
print(f"recall (neg, pos): {recall}")
print(f"f1 (neg, pos): {f1}")
print(roc_auc_score(y_test.astype(int), y_pred_proba[:,1]))

precision (neg, pos): [0.55045872 0.98698707]
recall (neg, pos): [0.52478134 0.9882494 ]
f1 (neg, pos): [0.53731343 0.98761783]
0.9109436648931584


In [31]:
for name, imp in zip(model.feature_names_, model.feature_importances_):
    print(name, imp)

function 1.106323898069242
landuse 27.600955632331644
area 16.274123295603832
volume 12.037450314143248
population_density 42.981146859852046


## Infer the model; fill building types

In [32]:
X_infer = to_predict[all_features]
for feature in categorical_features:
    X_infer.loc[:,feature] = X_infer[feature].fillna('')

X_infer = pd.DataFrame(preprocess.fit_transform(X_infer), columns=X_infer.columns)
X_infer.head(3)

,function,landuse,area,volume,population_density
0,,farmyard,0.515927,0.02937,-0.404689
1,,farmyard,0.555271,0.035194,-0.404689
2,,farmyard,0.616822,0.042701,-0.404689


In [36]:
y_infer = model.predict_proba(X_infer)
to_predict.loc[:, 'target'] = (y_infer[:, 1] > confidence).astype(int)
to_predict.head(2)

,height,building_type,name,function,lat,lon,landuse,geometry,index_right,population_density,income_density,area,perimeter,volume,slenderness,compactness,target
0,5.671275,yes,None,None,20.485226,44.937775,farmyard,"POLYGON ((459383.657 4976183.351, 459398.318 4...",11.0,0.000409,50.666154,466.166466,89.959844,41936.262676,0.262670,0.723858,0
1,8.052044,yes,None,None,20.485714,44.937992,farmyard,"POLYGON ((459421.482 4976207.996, 459437.618 4...",11.0,0.000409,50.666154,491.851058,93.379530,45928.820735,0.363069,0.708827,0


### Let's check the residential tag distribution by landuse

In [37]:
residential_by_landuse = dataset.groupby('landuse').aggregate({'target': ['mean', 'count']})
to_predict.groupby('landuse').aggregate({'target': ['mean', 'count']}).join(residential_by_landuse, rsuffix='_initial').sort_values(by=[('target', 'count')], ascending=False).head(10)

target         target_initial         
                  mean   count           mean    count
landuse                                               
residential   0.996958  120966       0.989531  69061.0
industrial    0.049688    6098       0.363964   1665.0
farmland      0.986599    1567       0.942149    242.0
commercial    0.009421    1486       0.407263    771.0
forest        0.991198    1477       0.970414    169.0
farmyard      0.120656    1036       0.353659    164.0
cemetery      0.977330     794       0.944444     36.0
grass         0.586667     750       0.687500    128.0
construction  0.760726     606       0.842466    146.0
military      0.930192     573       0.979866    298.0

### Merge the results with the data

In [72]:
buildings['is_residential'] = pd.concat([to_predict[['target']], dataset[['target']]], axis=0)
buildings['is_residential'] = buildings['is_residential'].astype(bool)
buildings_proj['is_residential'] = buildings['is_residential']

In [70]:
buildings.tail()

,height,building_type,name,function,lat,lon,landuse,geometry,target,is_residential
441217,4.391817,None,None,None,20.405135,44.999977,residential,"POLYGON ((20.40514 44.99995, 20.40517 44.99999...",1.0,True
441218,3.852985,None,None,None,20.403149,45.000058,industrial,"POLYGON ((20.40332 44.99992, 20.40338 44.99998...",0.0,False
441219,1.921309,None,None,None,20.402714,44.999960,industrial,"POLYGON ((20.40284 44.99993, 20.40265 45.00004...",0.0,False
441220,2.428521,None,None,None,20.304300,45.000090,None,"POLYGON ((20.3041 45.00002, 20.30415 44.99998,...",NaN,True
441221,5.875709,None,None,None,20.301300,45.000005,residential,"POLYGON ((20.30135 45, 20.30128 45.00004, 20.3...",NaN,True


### Visualize them on a map

In [85]:
def _get_color(is_residential, inferred=False):
    if not inferred:
        return '#FB8C00' if is_residential else '#303F9F'
    return '#D84315' if is_residential else '#64B5F6'

def _get_color_inferred(is_residential):
    return _get_color(is_residential, True)

buildings_proj['color'] = pd.concat([to_predict[['target']].map(_get_color_inferred), dataset[['target']].map(_get_color)], axis=0)


In [91]:
import matplotlib.pyplot as plt
import contextily as ctx
import shapely

fig, ax = plt.subplots(figsize=(40, 40), dpi=400)
#fig.patch.set_facecolor("black")

buildings_proj = buildings_proj[~buildings_proj.geometry.map(lambda x: isinstance(x, shapely.Point))]
buildings_proj.plot(
    ax=ax,
    color=buildings_proj["color"],
    edgecolor="none"
)
ax.set_title("Resident and non-resident buildings")
ax.legend()

ctx.add_basemap(
    ax,
    source=ctx.providers.CartoDB.DarkMatter,
    alpha=0.2
)

ax.set_axis_off()
fig.savefig("images/belgrade_res_nonres_with_prediction.png", dpi=400, bbox_inches="tight", pad_inches=0)
plt.close(fig)

/var/folders/yz/1n0b98zd7zqcxhyl8wl435h40000gn/T/ipykernel_18158/3921884525.py:15: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()
